In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/KeepCoding/NLP/Práctica

/content/drive/MyDrive/KeepCoding/NLP/Práctica


In [4]:
import pandas as pd

In [12]:
df_reviews_filtered = pd.read_csv("df_reviews_filtered.csv")

In [13]:
df_reviews_filtered["reviewText"].isna().sum()

np.int64(0)

In [15]:
reviews_dict=df_reviews_filtered[['reviewText','sentiment_label']].to_dict("index")

In [16]:
import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import TreebankWordTokenizer
from nltk import ngrams
from nltk.probability import FreqDist

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [17]:
# Stopwords estándar
custom_stopwords = set(stopwords.words("english"))

# Conservamos negaciones importantes para análisis de sentimiento
custom_stopwords.discard("no")
custom_stopwords.discard("not")
custom_stopwords.discard("nor")

# Añadimos stopwords detectadas durante la exploración
custom_stopwords.update(["use", "used", "one"])

In [18]:
def review_to_words(review):

    # Convertimos a minúscula y quitamos todo lo que no sea texto o números
    text = re.sub(r"[^a-zA-Z0-9]", " ", review.lower())
    # Dividimos en tokens por espacios
    tokenizer = TreebankWordTokenizer()
    words = tokenizer.tokenize(text)
    # Eliminamos stopwords
    words = [w for w in words if w not in custom_stopwords]

    return words

#### Para el preprocesado del texto se ha implementado una función encargada de normalizar y limpiar las reviews antes del entrenamiento de los modelos.

#### En primer lugar, el texto se convierte a minúsculas con el objetivo de evitar duplicados artificiales entre palabras como “Good” y “good”.

#### Posteriormente, se eliminan caracteres especiales y signos de puntuación mediante expresiones regulares, conservando únicamente letras y números.

#### Tras ello, se realiza una tokenización a nivel de palabra utilizando TreebankWordTokenizer de NLTK, obteniendo una separación de tokens más robusta que un simple split por espacios.

#### Finalmente, se eliminan stopwords poco informativas utilizando las stopwords estándar de NLTK junto con algunas adicionales detectadas durante la exploración inicial del corpus.

#### No obstante, se han conservado negaciones importantes como “not” o “no”, ya que pueden modificar completamente el significado y sentimiento de una review.

In [19]:
review_to_words(df_reviews_filtered["reviewText"][3])

['good',
 'computer',
 'rating',
 'impacted',
 'user',
 'interface',
 'feels',
 'like',
 'engineered',
 'first',
 'days',
 'vcrs',
 'digital',
 'watches',
 'programming',
 'pain',
 'easy',
 'accidentally',
 'clear']

In [20]:
bigrams_ = []
trigrams_ = []

for review in df_reviews_filtered["reviewText"].dropna():
    words = review_to_words(review)

    bigrams_.extend(list(ngrams(words, 2)))
    trigrams_.extend(list(ngrams(words, 3)))

bg_freq = FreqDist(bigrams_)
tg_freq = FreqDist(trigrams_)

In [21]:
bg_freq.most_common(10)

[(('well', 'made'), 343),
 (('would', 'not'), 340),
 (('would', 'recommend'), 246),
 (('works', 'great'), 238),
 (('works', 'well'), 236),
 (('not', 'sure'), 232),
 (('much', 'better'), 217),
 (('good', 'quality'), 190),
 (('not', 'good'), 177),
 (('make', 'sure'), 177)]

In [22]:
tg_freq.most_common(10)

[(('would', 'not', 'recommend'), 53),
 (('would', 'recommend', 'anyone'), 44),
 (('last', 'long', 'time'), 44),
 (('would', 'not', 'buy'), 40),
 (('red', 'dot', 'sight'), 37),
 (('could', 'not', 'get'), 34),
 (('ruger', '10', '22'), 31),
 (('not', 'big', 'deal'), 29),
 (('gets', 'job', 'done'), 26),
 (('not', 'much', 'say'), 26)]

#### Inicialmente, los n-grams se calcularon concatenando todas las reviews (como anteriormente) en un único texto completo. Aunque este enfoque permitía obtener una primera aproximación rápida, presentaba el inconveniente de generar posibles combinaciones de palabras entre reviews distintas.

#### Posteriormente, el proceso fue mejorado generando los n-grams review por review de forma independiente, evitando así la creación de relaciones artificiales entre el final de una review y el comienzo de otra.

#### Tras aplicar el preprocesado, los n-grams obtenidos resultan mucho más representativos semánticamente y contienen patrones relacionados con opiniones reales de los usuarios.

#### A diferencia de la exploración inicial, donde predominaban signos de puntuación y tokens poco relevantes, ahora aparecen expresiones útiles para el análisis de sentimiento como “works great”, “good quality” o “highly recommend”.

#### Además, la conservación de negaciones ha permitido mantener patrones negativos relevantes como “would not” o “not fit”, que podrían aportar información importante durante el entrenamiento del modelo.